#  DataRetail 360: Pipeline ETL Master & Arquitectura Analítica
**Asignatura:** Estructuras y Visualización de Datos (IDY1102)  
**Caso Semestral:** DataRetail 360  

---
###  Guía de Ejecución en Colab
1. Ve al menú superior: **Entorno de ejecución → Ejecutar todo** (o presiona `Ctrl + F9`).
2. El cuadernillo se encuentra dividido en **4 Bloques Técnicos**, asignados a cada integrante del equipo para la presentación oral y demostración práctica.


In [23]:
# Este primer bloque es un experimento para ver si logro persistencia en Google Drive
import os
from google.colab import drive
#montamos
drive.mount('/content/drive')
#ruta de la carpeta compartida
RUTA_PROYECTO = '/content/drive/MyDrive/EJECUTABLES'
#cambiamos el directorio de trabajo
if os.path.exists(RUTA_PROYECTO):
  os.chdir(RUTA_PROYECTO)
  print("Carpeta del proyecto en Drive conectadas:")
  print("Directorio actual:", os.getcwd())
else:
  print(f"No se encontró la carpeta en Drive: {RUTA_PROYECTO}")
  print("Asegurate de haber creado el acceso directo a 'EJECUTABLES' en 'Mi unidad'")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Carpeta del proyecto en Drive conectadas:
Directorio actual: /content/drive/MyDrive/EJECUTABLES


In [24]:
# --- INSTALACIÓN DE DEPENDENCIAS (GOOGLE COLAB) ---
!pip install -q pandera tenacity pyarrow


##  BLOQUE 1: Modelado Dimensional y Evolución de Datos (SCD Tipo 2)
 Presentación de la Problemática, Declaración del Grano Transaccional, Esquema Estrella DDL, demostración de SCD Tipo 2 en Producto y Cliente, y definición de la Capa Semántica para IA.


In [25]:
# --- BLOQUE 1.1: CONFIGURACIÓN E INICIALIZACIÓN DE LA BD DWH ---
import sqlite3
import pandas as pd
import numpy as np

# Conexión a SQLite con integridad referencial estricta
con = sqlite3.connect('dataretail360_dwh.db')
con.execute("PRAGMA foreign_keys = ON;")
cur = con.cursor()

# 1. CREACIÓN DEL ESQUEMA ESTRELLA (DDL: 6 Dimensiones + 1 Tabla de Hechos)
cur.executescript('''
DROP VIEW IF EXISTS vw_ventas_dashboard;
DROP VIEW IF EXISTS mv_ventas_mes_region;
DROP TABLE IF EXISTS fact_ventas;
DROP TABLE IF EXISTS tbl_rechazos_auditoria;
DROP TABLE IF EXISTS stg_ventas;
DROP TABLE IF EXISTS dim_fecha;
DROP TABLE IF EXISTS dim_sucursal;
DROP TABLE IF EXISTS dim_geografia;
DROP TABLE IF EXISTS dim_canal;
DROP TABLE IF EXISTS dim_cliente;
DROP TABLE IF EXISTS dim_producto;

-- Dimensiones Base
CREATE TABLE dim_fecha (
    fecha_key INTEGER PRIMARY KEY,
    fecha TEXT UNIQUE NOT NULL,
    dia INTEGER NOT NULL,
    mes INTEGER NOT NULL,
    nombre_mes TEXT NOT NULL,
    trimestre INTEGER NOT NULL,
    anio INTEGER NOT NULL,
    dia_semana TEXT NOT NULL,
    es_fin_de_semana INTEGER NOT NULL
);

CREATE TABLE dim_geografia (
    geografia_key INTEGER PRIMARY KEY AUTOINCREMENT,
    comuna TEXT UNIQUE NOT NULL,
    region TEXT NOT NULL,
    pais TEXT NOT NULL
);

CREATE TABLE dim_sucursal (
    sucursal_key INTEGER PRIMARY KEY AUTOINCREMENT,
    sucursal TEXT UNIQUE NOT NULL,       -- Clave natural que viene en los CSV
    nombre_sucursal TEXT NOT NULL,       -- Etiqueta legible para Dashboards
    comuna TEXT NOT NULL,
    region TEXT NOT NULL
);

CREATE TABLE dim_canal (
    canal_key INTEGER PRIMARY KEY AUTOINCREMENT,
    canal TEXT UNIQUE NOT NULL
);

-- Dimensión Cliente con soporte SCD Tipo 2 y seudonimización (Ley 21.719)
CREATE TABLE dim_cliente (
    cliente_key INTEGER PRIMARY KEY AUTOINCREMENT,
    id_cliente_hash TEXT NOT NULL,       -- Clave Natural seudonimizada (SHA-256)
    region TEXT NOT NULL,
    valido_desde TEXT NOT NULL,
    valido_hasta TEXT,
    es_actual INTEGER NOT NULL           -- 1 = Vigente, 0 = Histórico
);

-- Dimensión Producto con soporte SCD Tipo 2
CREATE TABLE dim_producto (
    producto_key INTEGER PRIMARY KEY AUTOINCREMENT,
    producto_id INTEGER NOT NULL,
    sku TEXT NOT NULL,
    producto TEXT NOT NULL,
    categoria TEXT NOT NULL,
    precio_unitario REAL NOT NULL,
    valido_desde TEXT NOT NULL,
    valido_hasta TEXT,
    es_actual INTEGER NOT NULL           -- 1 = Vigente, 0 = Histórico
);

-- Tabla de Hechos (Grano: 1 línea de venta por producto, cliente, sucursal, geografía, canal y fecha)
CREATE TABLE fact_ventas (
    id_venta INTEGER PRIMARY KEY,
    fecha_key INTEGER NOT NULL,
    producto_key INTEGER NOT NULL,
    sucursal_key INTEGER NOT NULL,
    geografia_key INTEGER NOT NULL,
    cliente_key INTEGER NOT NULL,
    canal_key INTEGER NOT NULL,
    estado TEXT NOT NULL,                -- OK / DEVUELTA
    cantidad INTEGER NOT NULL,
    monto_bruto REAL NOT NULL,
    monto_neto REAL NOT NULL,
    FOREIGN KEY (fecha_key) REFERENCES dim_fecha(fecha_key),
    FOREIGN KEY (producto_key) REFERENCES dim_producto(producto_key),
    FOREIGN KEY (sucursal_key) REFERENCES dim_sucursal(sucursal_key),
    FOREIGN KEY (geografia_key) REFERENCES dim_geografia(geografia_key),
    FOREIGN KEY (cliente_key) REFERENCES dim_cliente(cliente_key),
    FOREIGN KEY (canal_key) REFERENCES dim_canal(canal_key)
);
''')
con.commit()
print("Esquema Estrella DDL creado exitosamente (6 Dimensiones + 1 Tabla de Hechos, FK activas).")

Esquema Estrella DDL creado exitosamente (6 Dimensiones + 1 Tabla de Hechos, FK activas).


In [26]:
# --- BLOQUE 1.2: CARGA MAESTRA REAL (DESDE JSON) Y CORTE SCD TIPO 2 ---
import glob
import json
import hashlib

SALT_SEUDONIMO = os.getenv("SALT_SEUDONIMO", "salt_demo_academico")
RUTA_LANDING = "data/landing"
RUTA_CATALOGO = "data/landing/catalogo/productos.json"
RUTA_SUCURSALES = "data/landing/catalogo/sucursales.json"


def seudonimizar_rut(rut: str, salt: str) -> str:
    """Cumplimiento Ley 21.719 de Protección de Datos Personales (SHA-256 + Salt)."""
    if pd.isna(rut) or not str(rut).strip():
        return None
    clean_rut = str(rut).replace(".", "").replace("-", "").upper().strip()
    return hashlib.sha256(f"{salt}|{clean_rut}".encode()).hexdigest()[:16]


# 1. Carga maestra real desde JSON
assert os.path.exists(RUTA_SUCURSALES), f"Falta archivo: {RUTA_SUCURSALES}"
assert os.path.exists(RUTA_CATALOGO), f"Falta archivo: {RUTA_CATALOGO}"

with open(RUTA_SUCURSALES, 'r', encoding='utf-8') as f:
    sucursales_data = json.load(f)
with open(RUTA_CATALOGO, 'r', encoding='utf-8') as f:
    cat_productos = json.load(f)

skus_validos = {p['sku'].strip().upper() for p in cat_productos}

cur.executemany(
    'INSERT OR IGNORE INTO dim_geografia (comuna, region, pais) VALUES (?, ?, ?)',
    [(s['comuna'], s['region'], s['pais']) for s in sucursales_data]
)
cur.executemany(
    'INSERT OR IGNORE INTO dim_sucursal (sucursal, nombre_sucursal, comuna, region) VALUES (?, ?, ?, ?)',
    [(s['sucursal'], s['nombre_sucursal'], s['comuna'], s['region']) for s in sucursales_data]
)
cur.executemany(
    'INSERT OR IGNORE INTO dim_canal (canal) VALUES (?)',
    [('Tienda física',), ('E-commerce',), ('App',)]
)

# 2. Carga inicial de productos en dim_producto
for idx_p, prod in enumerate(cat_productos, start=101):
    # AU-002 parte en 'Novedades' para registrar su reclasificación a 'Accesorios' el 2025-02-15
    categoria_v1 = 'Novedades' if prod['sku'] == 'AU-002' else prod['categoria']
    cur.execute('''
        INSERT INTO dim_producto (producto_id, sku, producto, categoria, precio_unitario, valido_desde, valido_hasta, es_actual)
        VALUES (?, ?, ?, ?, ?, '2024-01-01', NULL, 1)
    ''', (idx_p, prod['sku'], prod['producto'], categoria_v1, prod['precio']))

# 3. Registro del cambio histórico SCD Tipo 2 (corte 2025-02-15)
# a) Producto AU-002 pasa de 'Novedades' a 'Accesorios'
prod_au = next(p for p in cat_productos if p['sku'] == 'AU-002')
prod_id_au = cur.execute("SELECT producto_id FROM dim_producto WHERE sku='AU-002' AND es_actual=1").fetchone()[0]
cur.execute("UPDATE dim_producto SET valido_hasta='2025-02-15', es_actual=0 WHERE sku='AU-002' AND es_actual=1")
cur.execute('''
    INSERT INTO dim_producto (producto_id, sku, producto, categoria, precio_unitario, valido_desde, valido_hasta, es_actual)
    VALUES (?, 'AU-002', ?, ?, ?, '2025-02-15', NULL, 1)
''', (prod_id_au, prod_au['producto'], prod_au['categoria'], prod_au['precio']))

# b) Cliente real del primer CSV (seudonimizado) se muda de Metropolitana a Biobío el 2025-02-15
csvs_disp = sorted(glob.glob(os.path.join(RUTA_LANDING, "ventas_*.csv")))
assert csvs_disp, f"No se encontraron CSVs en '{RUTA_LANDING}'."
rut_demo = pd.read_csv(csvs_disp[0], dtype=str, nrows=1)['rut_cliente'].iloc[0]
hash_demo_cliente = seudonimizar_rut(rut_demo, SALT_SEUDONIMO)

cur.execute(
    "INSERT INTO dim_cliente (id_cliente_hash, region, valido_desde, valido_hasta, es_actual) VALUES (?, 'Metropolitana', '2024-01-01', '2025-02-15', 0)",
    (hash_demo_cliente,)
)
cur.execute(
    "INSERT INTO dim_cliente (id_cliente_hash, region, valido_desde, valido_hasta, es_actual) VALUES (?, 'Biobío', '2025-02-15', NULL, 1)",
    (hash_demo_cliente,)
)
con.commit()

print(f"Dimensiones maestras cargadas: {len(sucursales_data)} sucursales, {len(cat_productos)} SKUs y corte SCD Tipo 2 registrado (2025-02-15).")
display(pd.read_sql_query("SELECT * FROM dim_sucursal", con))

Dimensiones maestras cargadas: 6 sucursales, 7 SKUs y corte SCD Tipo 2 registrado (2025-02-15).


,sucursal_key,sucursal,nombre_sucursal,comuna,region
0,1,Providencia,Sucursal Providencia,Providencia,Metropolitana
1,2,Las Condes,Sucursal Las Condes,Las Condes,Metropolitana
2,3,Maipú,Sucursal Maipú,Maipú,Metropolitana
3,4,Ñuñoa,Sucursal Ñuñoa,Ñuñoa,Metropolitana
4,5,Concepción,Sucursal Concepción,Concepción,Biobío
5,6,Antofagasta,Sucursal Antofagasta,Antofagasta,Antofagasta


In [27]:
# --- BLOQUE 1.3: DEFINICIÓN DE CAPA SEMÁNTICA PARA IA (NLQ / Text-to-SQL - IL 3.5) ---

CAPA_SEMANTICA_IA = """
Esquema DWH SQLite (DataRetail 360 - Ene a Mar 2025):
- Vista analítica principal: vw_ventas_dashboard
- Esquema estrella: fact_ventas (id_venta, fecha_key, producto_key, sucursal_key, geografia_key, cliente_key, canal_key, estado, cantidad, monto_bruto, monto_neto)
  + dim_fecha, dim_producto (SCD2), dim_sucursal, dim_geografia, dim_canal, dim_cliente (SCD2, solo id_cliente_hash).

Reglas de negocio obligatorias:
1. Venta neta: SUM(monto_neto). Ya excluye devoluciones (monto_neto = 0 cuando estado = 'DEVUELTA').
2. Pérdida por devoluciones: SUM(monto_bruto - monto_neto).
3. Tasa de devolución (%): 100.0 * SUM(CASE WHEN estado = 'DEVUELTA' THEN 1 ELSE 0 END) / COUNT(*).
4. Ticket promedio neto: SUM(monto_neto) / SUM(CASE WHEN estado = 'OK' THEN 1 ELSE 0 END).
5. Participación digital (%): 100.0 * SUM(CASE WHEN canal IN ('E-commerce', 'App') THEN monto_neto ELSE 0 END) / SUM(monto_neto).
6. SCD Tipo 2 (es_actual): usar es_actual = 1 SOLO para consultar catálogo o estado vigente actual; NO filtrar es_actual = 1 al consultar historia en fact_ventas.
7. Privacidad (Ley 21.719): dim_cliente expone únicamente id_cliente_hash (SHA-256); no existen RUTs ni nombres personales.
"""

print("Capa semántica definida y lista en la variable 'CAPA_SEMANTICA_IA':")
print(CAPA_SEMANTICA_IA)

Capa semántica definida y lista en la variable 'CAPA_SEMANTICA_IA':

Esquema DWH SQLite (DataRetail 360 - Ene a Mar 2025):
- Vista analítica principal: vw_ventas_dashboard
- Esquema estrella: fact_ventas (id_venta, fecha_key, producto_key, sucursal_key, geografia_key, cliente_key, canal_key, estado, cantidad, monto_bruto, monto_neto)
  + dim_fecha, dim_producto (SCD2), dim_sucursal, dim_geografia, dim_canal, dim_cliente (SCD2, solo id_cliente_hash).

Reglas de negocio obligatorias:
1. Venta neta: SUM(monto_neto). Ya excluye devoluciones (monto_neto = 0 cuando estado = 'DEVUELTA').
2. Pérdida por devoluciones: SUM(monto_bruto - monto_neto).
3. Tasa de devolución (%): 100.0 * SUM(CASE WHEN estado = 'DEVUELTA' THEN 1 ELSE 0 END) / COUNT(*).
4. Ticket promedio neto: SUM(monto_neto) / SUM(CASE WHEN estado = 'OK' THEN 1 ELSE 0 END).
5. Participación digital (%): 100.0 * SUM(CASE WHEN canal IN ('E-commerce', 'App') THEN monto_neto ELSE 0 END) / SUM(monto_neto).
6. SCD Tipo 2 (es_actual): usar

##  BLOQUE 2: Extracción Robusta, Homologación y Reglas de Calidad DAMA
 Ingesta de fuentes heterogéneas, Módulo Extract con reintentos (`tenacity`), Seudonimización de RUT con SHA-256 (Ley 21.719), Homologación de SKUs/canales y Aislamiento en Tabla de Rechazos.


In [28]:
# --- BLOQUE 2.1: EXTRACCIÓN ROBUSTA CON REINTENTOS (TENACITY) Y LANDING ZONE ---
from tenacity import retry, stop_after_attempt, wait_fixed


@retry(stop=stop_after_attempt(3), wait=wait_fixed(1), reraise=True)
def leer_csv_robusto(ruta_archivo: str) -> pd.DataFrame:
    """Lee un CSV con hasta 3 reintentos ante fallos transitorios de lectura (I/O Drive)."""
    tmp = pd.read_csv(ruta_archivo, dtype=str)
    tmp['archivo_origen'] = os.path.basename(ruta_archivo)
    return tmp


archivos_csv = sorted(glob.glob(os.path.join(RUTA_LANDING, "ventas_*.csv")))
assert archivos_csv, f"No se encontraron archivos CSV en '{RUTA_LANDING}'."

print(f"Leyendo {len(archivos_csv)} archivos CSV desde '{RUTA_LANDING}' con tenacity...")
df_raw = pd.concat([leer_csv_robusto(f) for f in archivos_csv], ignore_index=True)

for col in ['monto', 'cantidad', 'id_venta']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')

print(f"Extracción completada: {len(df_raw):,} filas crudas recibidas | SKUs válidos del catálogo: {len(skus_validos)}.")
display(df_raw.head(5))

Leyendo 90 archivos CSV desde 'data/landing' con tenacity...
Extracción completada: 101,000 filas crudas recibidas | SKUs válidos del catálogo: 7.


,id_venta,fecha,sku,sucursal,canal,cantidad,monto,estado,rut_cliente,archivo_origen
0,72221,2025-01-01,AU-002,Maipú,Tienda física,1,30000,OK,21.315.296-3,ventas_2025-01-01.csv
1,8163,2025-01-01,AU-002,Maipú,E-commerce,2,60000,OK,23.066.318-1,ventas_2025-01-01.csv
2,67800,2025-01-01,NB-001,Concepción,Tienda física,3,2397000,OK,22.351.434-0,ventas_2025-01-01.csv
3,38234,2025-01-01,CE-004,Antofagasta,App,2,698000,OK,17.002.348-K,ventas_2025-01-01.csv
4,34595,2025-01-01,NB-001,Maipú,Tienda física,3,2397000,OK,24.331.726-6,ventas_2025-01-01.csv


In [29]:
# --- BLOQUE 2.2: HOMOLOGACIÓN, SEUDONIMIZACIÓN Y REGLAS DE CALIDAD DAMA ---

def procesar_calidad_bloque2(df_in, salt, skus_catalogo):
    df = df_in.copy()

    # 1. Homologación (Consistencia)
    df['sku'] = df['sku'].fillna('').str.strip().str.upper()
    df['sucursal'] = df['sucursal'].fillna('').str.strip()
    df['estado'] = df['estado'].fillna('').str.strip().str.upper()
    df['canal'] = df['canal'].fillna('').str.strip().replace({
        'Tienda fisica': 'Tienda física',
        'ecommerce': 'E-commerce',
        'APP': 'App'
    })
    df['fecha'] = pd.to_datetime(df['fecha'].astype(str).str.replace('/', '-', regex=False), errors='coerce')

    # 2. Seudonimización de RUT (Privacidad - Ley 21.719) y eliminación de PII en texto plano
    df['id_cliente_hash'] = df['rut_cliente'].apply(lambda r: seudonimizar_rut(r, salt))
    df = df.drop(columns=['rut_cliente'])

    # 3. Reglas DAMA vectorizadas (Completitud, Validez y Consistencia)
    motivos = pd.Series('', index=df.index)
    motivos += np.where(df['sucursal'] == '', 'sin_sucursal,', '')
    motivos += np.where(df['monto'].isna() | (df['monto'] < 0), 'monto_negativo,', '')
    motivos += np.where(~df['sku'].isin(skus_catalogo), 'sku_desconocido,', '')
    motivos += np.where(df['fecha'].isna(), 'fecha_invalida,', '')
    motivos += np.where(df['cantidad'].isna() | (df['cantidad'] <= 0), 'cantidad_invalida,', '')
    motivos += np.where(df['id_cliente_hash'].isna(), 'sin_cliente,', '')
    df['motivo_rechazo'] = motivos.str.rstrip(',').replace('', 'OK')

    rechazos = df[df['motivo_rechazo'] != 'OK'].copy()
    limpios = df[df['motivo_rechazo'] == 'OK'].copy()

    # 4. Regla de Unicidad DAMA (Deduplicación por id_venta)
    dup_mask = limpios.duplicated(subset=['id_venta'], keep='first')
    if dup_mask.any():
        dup_rows = limpios[dup_mask].assign(motivo_rechazo='duplicado')
        rechazos = pd.concat([rechazos, dup_rows], ignore_index=True)
        limpios = limpios[~dup_mask].copy()

    return limpios, rechazos


limpios_b2, rechazos_b2 = procesar_calidad_bloque2(df_raw, SALT_SEUDONIMO, skus_validos)

print(f"Procesamiento finalizado. Filas Validadas: {len(limpios_b2):,} | Filas Rechazadas: {len(rechazos_b2):,}")
print("Verificación Ley 21.719 -> ¿'rut_cliente' eliminado de Plata/Auditoría?:", 'rut_cliente' not in limpios_b2.columns)
print("\n--- Resumen de Rechazos (aislados para auditoría) ---")
display(rechazos_b2['motivo_rechazo'].value_counts().to_frame('cantidad_filas'))

Procesamiento finalizado. Filas Validadas: 98,510 | Filas Rechazadas: 2,490
Verificación Ley 21.719 -> ¿'rut_cliente' eliminado de Plata/Auditoría?: True

--- Resumen de Rechazos (aislados para auditoría) ---


,cantidad_filas
motivo_rechazo,
duplicado,983
sin_sucursal,801
monto_negativo,497
sku_desconocido,197
"sin_sucursal,monto_negativo",7
"sin_sucursal,sku_desconocido",3
"monto_negativo,sku_desconocido",2


##  BLOQUE 3: Carga Incremental, Idempotencia y Patrón WAP
 Cálculo vectorizado de métricas (`monto_neto`), Contrato de datos con **Pandera**, Implementación del patrón Write–Audit–Publish (WAP), `UPSERT` Idempotente y control por `watermark`.


In [30]:
# --- BLOQUE 3.1: DERIVACIÓN VECTORIZADA Y CONTRATO PANDERA ---
import pandera.pandas as pa

limpios_b3 = limpios_b2.copy()

# 1. Derivación vectorizada de métricas y llaves temporales
limpios_b3['id_venta'] = limpios_b3['id_venta'].astype(int)
limpios_b3['cantidad'] = limpios_b3['cantidad'].astype(int)
limpios_b3['monto_bruto'] = limpios_b3['monto'].astype(float)
limpios_b3['monto_neto'] = np.where(limpios_b3['estado'] == 'OK', limpios_b3['monto_bruto'], 0.0)
limpios_b3['fecha_key'] = (
    limpios_b3['fecha'].dt.year * 10000
    + limpios_b3['fecha'].dt.month * 100
    + limpios_b3['fecha'].dt.day
).astype(int)
limpios_b3['fecha_iso'] = limpios_b3['fecha'].dt.strftime('%Y-%m-%d')

# 2. Contrato de Datos con Pandera
schema_ventas = pa.DataFrameSchema({
    "id_venta": pa.Column(int, unique=True, nullable=False),
    "fecha": pa.Column("datetime64[ns]", nullable=False),
    "sku": pa.Column(str, pa.Check.isin(skus_validos)),
    "sucursal": pa.Column(str, pa.Check.str_length(min_value=1)),
    "canal": pa.Column(str, pa.Check.isin(["Tienda física", "E-commerce", "App"])),
    "estado": pa.Column(str, pa.Check.isin(["OK", "DEVUELTA"])),
    "cantidad": pa.Column(int, pa.Check.gt(0)),
    "monto_bruto": pa.Column(float, pa.Check.ge(0)),
    "monto_neto": pa.Column(float, pa.Check.ge(0)),
    "id_cliente_hash": pa.Column(str, nullable=False),
})

schema_ventas.validate(limpios_b3, lazy=True)
assert 'rut_cliente' not in limpios_b3.columns, "Violación de privacidad: rut_cliente presente."
print("Contrato de datos Pandera validado exitosamente sin violaciones.")

Contrato de datos Pandera validado exitosamente sin violaciones.


In [31]:
# --- BLOQUE 3.2: PATRÓN WRITE-AUDIT-PUBLISH (WAP) E IDEMPOTENCIA (UPSERT REAL) ---

# 1. WRITE: Cargar staging limpio y tabla de rechazos (ambas sin PII)
limpios_b3.to_sql('stg_ventas', con, if_exists='replace', index=False)
rechazos_b2.to_sql('tbl_rechazos_auditoria', con, if_exists='replace', index=False)
print("1. [WRITE] Datos escritos en staging (stg_ventas) y rechazos en tbl_rechazos_auditoria.")

# 2. AUDIT: Quality Gate (umbral máximo 5% de rechazo)
total_staging = cur.execute("SELECT COUNT(*) FROM stg_ventas").fetchone()[0]
tasa_rechazo = len(rechazos_b2) / len(df_raw)
UMBRAL_RECHAZO_MAX = 0.05

assert total_staging > 0, "Audit Fallado: Staging vacío."
assert tasa_rechazo <= UMBRAL_RECHAZO_MAX, f"Audit Fallado: Tasa ({tasa_rechazo:.2%}) supera umbral ({UMBRAL_RECHAZO_MAX:.0%})."
print(f"2. [AUDIT] Quality Gate Aprobado (Tasa de rechazo observada: {tasa_rechazo:.2%} <= {UMBRAL_RECHAZO_MAX:.0%}).")

# 2.5 Sincronización idempotente de dim_fecha y dim_cliente (sin alterar historial SCD2)
MESES_ES = {1:'Enero', 2:'Febrero', 3:'Marzo', 4:'Abril', 5:'Mayo', 6:'Junio',
            7:'Julio', 8:'Agosto', 9:'Septiembre', 10:'Octubre', 11:'Noviembre', 12:'Diciembre'}
DIAS_ES = {0:'Lunes', 1:'Martes', 2:'Miércoles', 3:'Jueves', 4:'Viernes', 5:'Sábado', 6:'Domingo'}

cur.executemany('''
    INSERT OR IGNORE INTO dim_fecha
    (fecha_key, fecha, dia, mes, nombre_mes, trimestre, anio, dia_semana, es_fin_de_semana)
    VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
''', [
    (int(f.year * 10000 + f.month * 100 + f.day), f.strftime('%Y-%m-%d'), int(f.day),
     int(f.month), MESES_ES[f.month], int((f.month - 1) // 3 + 1), int(f.year),
     DIAS_ES[f.dayofweek], int(f.dayofweek >= 5))
    for f in pd.Series(sorted(limpios_b3['fecha'].unique()))
])

mapa_region = dict(cur.execute("SELECT sucursal, region FROM dim_sucursal").fetchall())
cli_lote = (
    limpios_b3.assign(region=limpios_b3['sucursal'].map(mapa_region))
    .groupby('id_cliente_hash')['region']
    .agg(lambda x: x.mode().iloc[0])
    .reset_index()
)
clientes_existentes = {r[0] for r in cur.execute("SELECT DISTINCT id_cliente_hash FROM dim_cliente").fetchall()}
nuevos_clientes = cli_lote[~cli_lote['id_cliente_hash'].isin(clientes_existentes)]

cur.executemany(
    "INSERT INTO dim_cliente (id_cliente_hash, region, valido_desde, valido_hasta, es_actual) VALUES (?, ?, '2024-01-01', NULL, 1)",
    list(nuevos_clientes.itertuples(index=False, name=None))
)
con.commit()

# 3. PUBLISH: UPSERT idempotente a fact_ventas respetando vigencias SCD Tipo 2
cur.executescript('''
INSERT OR REPLACE INTO fact_ventas (
    id_venta, fecha_key, producto_key, sucursal_key, geografia_key,
    cliente_key, canal_key, estado, cantidad, monto_bruto, monto_neto
)
SELECT
    s.id_venta, s.fecha_key, p.producto_key, su.sucursal_key, g.geografia_key,
    c.cliente_key, ca.canal_key, s.estado, s.cantidad, s.monto_bruto, s.monto_neto
FROM stg_ventas s
JOIN dim_fecha f      ON f.fecha_key = s.fecha_key
JOIN dim_sucursal su  ON su.sucursal = s.sucursal
JOIN dim_geografia g  ON g.comuna = su.comuna
JOIN dim_canal ca     ON ca.canal = s.canal
JOIN dim_cliente c    ON c.id_cliente_hash = s.id_cliente_hash
                     AND s.fecha_iso >= c.valido_desde
                     AND (c.valido_hasta IS NULL OR s.fecha_iso < c.valido_hasta)
JOIN dim_producto p   ON p.sku = s.sku
                     AND s.fecha_iso >= p.valido_desde
                     AND (p.valido_hasta IS NULL OR s.fecha_iso < p.valido_hasta);
''')
con.commit()

cnt_fact = cur.execute("SELECT COUNT(*) FROM fact_ventas").fetchone()[0]
watermark_max = cur.execute("SELECT MAX(fecha_key) FROM fact_ventas").fetchone()[0]
fk_errors = cur.execute("PRAGMA foreign_key_check;").fetchall()

assert cnt_fact == total_staging, f"Descuadre en PUBLISH: staging={total_staging}, fact={cnt_fact}"
assert len(fk_errors) == 0, f"Violación de Foreign Key detectada: {fk_errors}"
print(f"3. [PUBLISH] Promoción exitosa a fact_ventas: {cnt_fact:,} filas (100% match, 0 errores FK, watermark={watermark_max}).")

1. [WRITE] Datos escritos en staging (stg_ventas) y rechazos en tbl_rechazos_auditoria.
2. [AUDIT] Quality Gate Aprobado (Tasa de rechazo observada: 2.47% <= 5%).
3. [PUBLISH] Promoción exitosa a fact_ventas: 98,510 filas (100% match, 0 errores FK, watermark=20250331).


## BLOQUE 4: Arquitectura Medallion, Optimización y Observabilidad
Persistencia en capas Medallion (Bronce, Plata, Oro en Parquet/SQLite), Optimización mediante Índices B-Tree y Vista Materializada `mv_ventas_mes_region`, y Logging Estructurado JSON (`logs/etl.jsonl`).


In [32]:
# --- BLOQUE 4.1: MEDALLION ARCHITECTURE (PARQUET) Y OPTIMIZACIÓN EN DWH ---

# 1. Capa Bronce (CSV crudo inmutable) y Capa Plata (Parquet optimizado sin PII)
os.makedirs('data/bronce', exist_ok=True)
os.makedirs('data/plata', exist_ok=True)
df_raw.to_csv('data/bronce/ventas_raw_2025_Q1.csv', index=False)
limpios_b3.to_parquet('data/plata/ventas_limpias_2025_Q1.parquet', index=False)

print(f"Capa Bronce (CSV Crudo): {os.path.getsize('data/bronce/ventas_raw_2025_Q1.csv'):,} bytes")
print(f"Capa Plata (Parquet Optimizado sin PII): {os.path.getsize('data/plata/ventas_limpias_2025_Q1.parquet'):,} bytes")

# 2. Capa Oro: Índices B-Tree y Vistas Analíticas para Dashboard e IA
cur.executescript('''
CREATE INDEX IF NOT EXISTS idx_fact_fecha     ON fact_ventas(fecha_key);
CREATE INDEX IF NOT EXISTS idx_fact_producto  ON fact_ventas(producto_key);
CREATE INDEX IF NOT EXISTS idx_fact_sucursal  ON fact_ventas(sucursal_key);
CREATE INDEX IF NOT EXISTS idx_fact_geografia ON fact_ventas(geografia_key);
CREATE INDEX IF NOT EXISTS idx_fact_cliente   ON fact_ventas(cliente_key);
CREATE INDEX IF NOT EXISTS idx_fact_canal     ON fact_ventas(canal_key);

DROP VIEW IF EXISTS mv_ventas_mes_region;
CREATE VIEW mv_ventas_mes_region AS
SELECT
    f.fecha_key / 100 AS anio_mes,
    d.nombre_mes,
    g.region,
    COUNT(f.id_venta) AS total_transacciones,
    SUM(CASE WHEN f.estado = 'DEVUELTA' THEN 1 ELSE 0 END) AS transacciones_devueltas,
    SUM(f.cantidad) AS total_unidades,
    SUM(f.monto_bruto) AS venta_bruta,
    SUM(f.monto_neto) AS venta_total_neta
FROM fact_ventas f
JOIN dim_fecha d     ON d.fecha_key = f.fecha_key
JOIN dim_geografia g ON g.geografia_key = f.geografia_key
GROUP BY anio_mes, d.nombre_mes, g.region;

DROP VIEW IF EXISTS vw_ventas_dashboard;
CREATE VIEW vw_ventas_dashboard AS
SELECT
    f.id_venta,
    d.fecha,
    d.anio,
    d.trimestre,
    d.mes,
    d.nombre_mes,
    d.dia_semana,
    d.es_fin_de_semana,
    p.sku,
    p.producto,
    p.categoria AS categoria_historica,
    p.precio_unitario,
    p.es_actual AS es_categoria_vigente,
    su.sucursal,
    su.nombre_sucursal,
    g.comuna,
    g.region,
    g.pais,
    ca.canal,
    c.id_cliente_hash,
    c.region AS region_historica_cliente,
    f.estado,
    f.cantidad,
    f.monto_bruto,
    f.monto_neto
FROM fact_ventas f
JOIN dim_fecha d      ON d.fecha_key = f.fecha_key
JOIN dim_producto p   ON p.producto_key = f.producto_key
JOIN dim_sucursal su  ON su.sucursal_key = f.sucursal_key
JOIN dim_geografia g  ON g.geografia_key = f.geografia_key
JOIN dim_canal ca     ON ca.canal_key = f.canal_key
JOIN dim_cliente c    ON c.cliente_key = f.cliente_key;
''')
con.commit()
print("Índices B-Tree y Vistas Capa Oro ('mv_ventas_mes_region', 'vw_ventas_dashboard') creados exitosamente.")

Capa Bronce (CSV Crudo): 9,395,991 bytes
Capa Plata (Parquet Optimizado sin PII): 1,150,713 bytes
Índices B-Tree y Vistas Capa Oro ('mv_ventas_mes_region', 'vw_ventas_dashboard') creados exitosamente.


In [33]:
# --- BLOQUE 4.2: CONSULTA RÁPIDA DE ORO, VALIDACIÓN SCD2 CON HECHOS REALES Y LOGGING JSONL ---
import uuid
from datetime import datetime, timezone

print("--- Vista Capa Oro: mv_ventas_mes_region (3 regiones x 3 meses) ---")
display(pd.read_sql_query("SELECT * FROM mv_ventas_mes_region", con))

print("\n--- Evidencia SCD Tipo 2 con Hechos Reales: Producto AU-002 ---")
display(pd.read_sql_query('''
SELECT p.producto_key, p.sku, p.categoria, p.valido_desde,
       COALESCE(p.valido_hasta, 'VIGENTE') AS valido_hasta, p.es_actual,
       COUNT(f.id_venta) AS total_ventas, SUM(f.monto_neto) AS monto_neto
FROM fact_ventas f
JOIN dim_producto p ON p.producto_key = f.producto_key
WHERE p.sku = 'AU-002'
GROUP BY p.producto_key;
''', con))

print("\n--- Evidencia SCD Tipo 2 con Hechos Reales: Cliente Seudonimizado ---")
display(pd.read_sql_query('''
SELECT c.cliente_key, c.id_cliente_hash, c.region, c.valido_desde,
       COALESCE(c.valido_hasta, 'VIGENTE') AS valido_hasta, c.es_actual,
       COUNT(f.id_venta) AS compras, SUM(f.monto_neto) AS monto_neto
FROM fact_ventas f
JOIN dim_cliente c ON c.cliente_key = f.cliente_key
WHERE c.id_cliente_hash = ?
GROUP BY c.cliente_key;
''', con, params=(hash_demo_cliente,)))

# Logging Estructurado JSON (logs/etl.jsonl)
os.makedirs('logs', exist_ok=True)
run_id = uuid.uuid4().hex[:8]

log_event = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "run_id": run_id,
    "level": "INFO",
    "step": "pipeline_complete",
    "rows_raw": len(df_raw),
    "rows_processed": len(limpios_b3),
    "rows_rejected": len(rechazos_b2),
    "rejection_rate": round(tasa_rechazo, 4),
    "watermark_fecha_key": int(watermark_max),
    "status": "SUCCESS",
}

with open('logs/etl.jsonl', 'a', encoding='utf-8') as f:
    f.write(json.dumps(log_event, ensure_ascii=False) + '\n')

print(f"\nEvento de corrida registrado en logs/etl.jsonl con run_id={run_id}:")
print(json.dumps(log_event, indent=2))

--- Vista Capa Oro: mv_ventas_mes_region (3 regiones x 3 meses) ---


,anio_mes,nombre_mes,region,total_transacciones,transacciones_devueltas,total_unidades,venta_bruta,venta_total_neta
0,202501,Enero,Antofagasta,5507,281,10984,2.874458e+09,2.716841e+09
1,202501,Enero,Biobío,5687,306,11212,2.900941e+09,2.761057e+09
2,202501,Enero,Metropolitana,22726,1104,45509,1.233386e+10,1.171611e+10
3,202502,Febrero,Antofagasta,5102,270,10062,2.620068e+09,2.482900e+09
4,202502,Febrero,Biobío,5064,256,10215,2.748651e+09,2.615012e+09
5,202502,Febrero,Metropolitana,20706,1055,41300,1.099573e+10,1.044077e+10
6,202503,Marzo,Antofagasta,5667,258,11332,2.984730e+09,2.856225e+09
7,202503,Marzo,Biobío,5636,280,11232,2.947917e+09,2.799340e+09
8,202503,Marzo,Metropolitana,22415,1086,44864,1.193942e+10,1.140282e+10



--- Evidencia SCD Tipo 2 con Hechos Reales: Producto AU-002 ---


,producto_key,sku,categoria,valido_desde,valido_hasta,es_actual,total_ventas,monto_neto
0,2,AU-002,Novedades,2024-01-01,2025-02-15,0,7064,398730000.0
1,8,AU-002,Accesorios,2025-02-15,VIGENTE,1,7061,397920000.0



--- Evidencia SCD Tipo 2 con Hechos Reales: Cliente Seudonimizado ---


,cliente_key,id_cliente_hash,region,valido_desde,valido_hasta,es_actual,compras,monto_neto
0,1,daa31187b0a49aa0,Metropolitana,2024-01-01,2025-02-15,0,30,13232000.0
1,2,daa31187b0a49aa0,Biobío,2025-02-15,VIGENTE,1,35,16616000.0



Evento de corrida registrado en logs/etl.jsonl con run_id=3bf5862e:
{
  "timestamp": "2026-11-24T02:07:54.146339+00:00",
  "run_id": "3bf5862e",
  "level": "INFO",
  "step": "pipeline_complete",
  "rows_raw": 101000,
  "rows_processed": 98510,
  "rows_rejected": 2490,
  "rejection_rate": 0.0247,
  "watermark_fecha_key": 20250331,
  "status": "SUCCESS"
}
